<a href="https://colab.research.google.com/github/ZavSan95/devtree/blob/main/Copia_de_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)

In [ ]:
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    n_classes=2,
    weights=[0.99, 0.01],
    n_clusters_per_class=1,
    class_sep=1.0,
    flip_y=0,
    random_state=42
)

print('Distribucion total:')
print(pd.Series(y).value_counts().sort_index())
print()
print('Porcentaje por clase:')
print((pd.Series(y).value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print('Entrenamiento:')
print(pd.Series(y_train).value_counts().sort_index())
print()
print('Prueba:')
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
modelo_dummy = DummyClassifier(strategy='most_frequent')
modelo_dummy.fit(X_train, y_train)

modelo_logistico = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
modelo_logistico.fit(X_train, y_train)

modelo_balanceado = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
)
modelo_balanceado.fit(X_train, y_train)

In [ ]:
pred_dummy = modelo_dummy.predict(X_test)
prob_dummy = modelo_dummy.predict_proba(X_test)[:, 1]

pred_logistico = modelo_logistico.predict(X_test)
prob_logistico = modelo_logistico.predict_proba(X_test)[:, 1]

prob_balanceado = modelo_balanceado.predict_proba(X_test)[:, 1]
pred_balanceado_050 = (prob_balanceado >= 0.50).astype(int)
pred_balanceado_020 = (prob_balanceado >= 0.20).astype(int)

In [ ]:
def evaluar_modelo(nombre, y_real, y_pred, probabilidades):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_real, y_pred),
        'Precision': precision_score(y_real, y_pred, zero_division=0),
        'Recall': recall_score(y_real, y_pred, zero_division=0),
        'F1': f1_score(y_real, y_pred, zero_division=0),
        'PR-AUC (AP)': average_precision_score(y_real, probabilidades),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }

resultados = pd.DataFrame([
    evaluar_modelo('Predictor mayoritario', y_test, pred_dummy, prob_dummy),
    evaluar_modelo('Regresion logistica', y_test, pred_logistico, prob_logistico),
    evaluar_modelo('Balanceada - umbral 0.50', y_test, pred_balanceado_050, prob_balanceado),
    evaluar_modelo('Balanceada - umbral 0.20', y_test, pred_balanceado_020, prob_balanceado),
])

resultados.round(4)

In [ ]:
modelos_pred = [
    ('Predictor mayoritario', pred_dummy),
    ('Regresion logistica', pred_logistico),
    ('Balanceada - umbral 0.50', pred_balanceado_050),
    ('Balanceada - umbral 0.20', pred_balanceado_020),
]

for nombre, predicciones in modelos_pred:
    ConfusionMatrixDisplay.from_predictions(y_test, predicciones, values_format='d')
    plt.title(nombre)
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for nombre, probabilidades in [
    ('Predictor mayoritario', prob_dummy),
    ('Regresion logistica', prob_logistico),
    ('Regresion balanceada', prob_balanceado),
]:
    precision, recall, _ = precision_recall_curve(y_test, probabilidades)
    ap = average_precision_score(y_test, probabilidades)
    plt.plot(recall, precision, label=f'{nombre} - AP={ap:.3f}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curvas Precision-Recall')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
resultados['Costo'] = 10 * resultados['FN'] + resultados['FP']
resultados[['Modelo', 'FP', 'FN', 'Costo']]